<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l9.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L9 · Checklist go-live
60 trades netos de costos: veredicto GO/NO-GO con adherencia y prueba de estrés.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l9.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l9.csv'), Path('data/c6_l9.csv'), Path('c6_l9.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
neto = df['ret_neto_bps']
bruto = df['ret_bruto_bps']
adh = df['checklist_ok'].mean()
print('bruto %+.2f · neto %+.2f bps/trade · winrate neto %.0f%%' % (bruto.mean(), neto.mean(), float((neto > 0).mean()) * 100))
print('adherencia %.0f%% (%d/60, umbral 70%%)' % (adh * 100, int(df.checklist_ok.sum())))

In [ ]:
s = (neto <= 0).astype(int)
peor, racha = 0, 0
for v in s:
    racha = racha + 1 if v else 0
    peor = max(peor, racha)
estres = (df.ret_bruto_bps - 8.0).mean()
print('peor racha perdedora: %d trades' % peor)
print('estres a 8 bps: neto %+.2f bps/trade' % estres)
veredicto = 'GO con stake minimo' if (neto.mean() > 0 and adh >= 0.70) else 'NO-GO'
print('veredicto:', veredicto)

In [ ]:
assert len(df) == 60 and (df.costo_bps == 4.0).all()
assert abs(df.ret_neto_bps.mean() - 5.78) < 0.6 and 0.50 <= float((df.ret_neto_bps > 0).mean()) <= 0.62
assert int(df.checklist_ok.sum()) == 44
assert (df.ret_bruto_bps - df.costo_bps - df.ret_neto_bps).abs().max() < 1e-9
print('OK L9: GO verificado, neto +5,78 bps con adherencia 73%')